# Exploring the isis search engine

Everything the web app does is plain Python in `isis.engine`, so you can call it, take it apart and change it here. The web app and this notebook can share the same index cache.

Setup, from the repository root:

```sh
pip install -e ".[notebook]"
python scripts/fetch_sample_corpus.py      # or point CORPUS below at your own folder
```

The first `refresh()` embeds every passage (about 15 passages/s on one slow core; much faster on a laptop). After that it only re-reads files that changed.

In [ ]:
from isis.engine.index import Engine
from isis.engine.semantic import Embedder
from isis.engine.related import RelatedConfig, TaskPrefs
from isis.engine.report import show_hits, show_related, show_table

CORPUS = "../sample_corpus"
engine = Engine(CORPUS, embedder=Embedder(), exclude=(f"{CORPUS}/_tasks",))
print(engine.refresh())
engine.status

## Typed search

BM25 over passages. `"quoted phrases"` must appear verbatim, and a half-typed last word is expanded to the corpus terms it starts. `mode="hybrid"` fuses BM25 with embedding similarity by reciprocal rank.

In [ ]:
show_hits(engine.search('"white whale"', limit=5))

In [ ]:
show_hits(engine.search("animals changing over many generations", limit=5, mode="hybrid"))

## The task document as the query

`related()` takes the text of a task document, the whole of which is the query, and returns four lanes:

- **Core**: strongest on both keywords and meaning (reciprocal-rank fusion).
- **Hidden connections**: close in meaning, little keyword overlap.
- **Bridges**: clearly related to two of the task's themes at once (themes are its headings, or clusters of its paragraphs).
- **Gaps**: related to the themes with the fewest collected excerpts.

Each result's `why` dict says how it got there: its semantic z-score, its BM25 rank, and lane-specific details.

In [ ]:
task = """# Whales and the sea
The sperm whale dives into the deep ocean to hunt giant squid, and the whalers chase it with harpoons.

# Variation and selection
Species vary, and natural selection preserves the variations that help an animal survive in its struggle for existence.
"""

result = engine.related(task)
show_related(result)

## Turn the knobs

Every threshold lives in `RelatedConfig` (see `isis/engine/related.py` for the full list with comments). The most useful ones:

| knob | default | what it does |
|---|---|---|
| `z_strong` | 2.5 | semantic z-score that gets a passage through the gate on its own |
| `hidden_max_obviousness` | 0.3 | how much keyword overlap a "hidden connection" may have (BM25 relative to the best hit) |
| `bridge_z` | 1.5 | z-score a passage needs on *both* of two themes to count as a bridge |
| `query` | `"document"` | `"document"`: the whole task is one query; `"paragraphs"`: each paragraph is a query, and a passage scores by its best match |
| `cursor_boost` | 1.0 | above 1, weights the paragraph at the cursor up (pass `cursor_line=` to `related()`) |
| `mmr_lambda` | 0.6–0.7 | relevance vs. variety when filling each lane |
| `total_slots` | 24 | results across all lanes |

`TaskPrefs` carries what the user told the app: pinned and banned terms, dismissed passages, and add/dismiss counts per lane.

In [ ]:
config = RelatedConfig(z_strong=3.0, hidden_max_obviousness=0.15, bridge_z=1.25)
prefs = TaskPrefs(bans=["squid"], pins=["harpoon"])
show_related(engine.related(task, config=config, prefs=prefs), per_lane=3)

# The same task with each paragraph as its own query: strong matches to either topic, not their overlap.
show_related(engine.related(task, config=RelatedConfig(query="paragraphs")), per_lane=3)

## Look inside

The steps are plain functions over numpy arrays. For example: how many passages stand out against the whole document, compared with each paragraph on its own? A document about two distant topics makes a blurrier query, so fewer passages stand out against it.

In [ ]:
import numpy as np
from isis.engine.related import task_units, zscores

units = task_units(task)
vecs = engine.embedder.embed([u.heading + "\n" + u.text for u in units])
sizes = np.array([len(u.text.split()) for u in units], np.float32)
document = (sizes[:, None] * vecs).sum(0)
document /= np.linalg.norm(document)

for name, v in [("whole document", document), *((f"paragraph {u.heading!r}", vecs[j]) for j, u in enumerate(units))]:
    _, z = zscores(engine.snapshot, v[None, :])
    print(f"{name}: {np.mean(z[:, 0] >= 2.5):.2%} of passages reach z >= 2.5; max z {z[:, 0].max():.1f}")

## Measure

`holdout` hides a share of a task's collected excerpts and checks how many come back as suggestions. `make_task` builds test tasks from the corpus until you have real ones. Compare configurations before changing a default.

In [ ]:
from isis.engine.evaluate import compare, make_task

tasks = [make_task(engine, q, n=8, seed=1) for q in ["whale hunting", "natural selection", "volcanic islands"]]
show_table(compare(engine, tasks, {
    "default": RelatedConfig(),
    "stricter gate": RelatedConfig(z_strong=3.0),
    "more hidden": RelatedConfig(hidden_max_obviousness=0.5),
}))

## Try another model

Any Hugging Face repo with ONNX weights and a `tokenizer.json` works; pooling is read from the repo. Give each model its own index folder so both caches survive.

In [ ]:
# bigger = Engine(CORPUS, embedder=Embedder("BAAI/bge-small-en-v1.5"), index_dir=f"{CORPUS}/.isis-bge-small",
#                 exclude=(f"{CORPUS}/_tasks",))
# bigger.refresh()
# show_table(compare(bigger, tasks, {"bge-small": RelatedConfig()}))